In [ ]:
import sys

PATH_TO_NOTEBOOK_DATA = "/notebook_data"
PATH_TO_DATA_QUALITY = "/notebook_data/qmeasures"

sys.path.append(PATH_TO_DATA_QUALITY)
sys.path.append(PATH_TO_NOTEBOOK_DATA)

In [ ]:
import torch
from torchvision.datasets import CIFAR10
import numpy as np
from tqdm import tqdm

In [ ]:
from qmeasures.metrics import IS

In [ ]:
inc_score = IS()

In [ ]:
dummy_probabilities = torch.rand((1000, 10))

for i in range(len(dummy_probabilities)):
    dummy_probabilities[i] = (1 / torch.sum(dummy_probabilities[i])) * dummy_probabilities[i]

sc = inc_score.instant_score(dummy_probabilities, already_transformed=True)
print(sc)

In [ ]:
trivial_probabilities = torch.Tensor(
    [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
)

sc = inc_score.instant_score(trivial_probabilities, feature_extractor=id_transform)
print("Should be 3:", sc)

In [ ]:
naive_probabilities = torch.Tensor(
    [[0.33, 0.33, 0.33], [0.33, 0.33, 0.33], [0.33, 0.33, 0.33]]
)

sc = inc_score.instant_score(naive_probabilities, feature_extractor=id_transform)
print("Should be 1:", sc)

In [ ]:
inc_score2 = IS()

In [ ]:
data = CIFAR10(root="./data",  download=True)

LIMIT = len(data)
N = 1000
L = 1000

index = np.random.choice(LIMIT, N, replace=False)
dataset = [data[i][0] for i in index]

In [ ]:
def is_unit_tensor(t):
    dim = None
    if len(tuple(t.size())) != 1:
        return False, dim
    if int(torch.count_nonzero(t)) != 1:
        return False, dim

    for i in range(len(t)):
        v = t[i]
        if v == 1:
            dim = i

        if v != 0 and v != 1:
            return False, dim

    return True, dim

In [ ]:
samples = inc_score2.transform_and_extract_features(dataset)

In [ ]:
dummy_large = torch.zeros(N, L)
little_n = min(L,N)
for i in range(little_n):
    dummy_large[i][i] = 1.

In [ ]:
l = []
for d in tqdm(dummy_large):
    res = is_unit_tensor(d)
    l.append(res[1])

l = list(set(l))
print(len(l))

In [ ]:
l2 = []
for s in tqdm(samples):
    res = is_unit_tensor(s)
    l2.append(res[1])

l2 = list(set(l2))
print(len(l2))

In [ ]:
score = inc_score2.run_score(dummy_large, already_transformed=True, times=50)
score

In [ ]:
score = inc_score2.run_score(samples, as_float=False, already_transformed=True, times=50)
score